In [0]:
# Function to connect ADLS with Databricks using Account Key method
def adls_connect():
    spark.conf.set(
    "fs.azure.account.key.adlsdataoptum.dfs.core.windows.net",
    dbutils.secrets.get(scope="optumScope", key="AdlsAccessKey"))
    return "ADLS Gen2 Connected Successfully"

In [0]:
# Function to list all the files in Bronze layer in ADLS Gen2
def list_bronze_files():
    display(dbutils.fs.ls("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/bronze"))
    return "Bronze files listed successfully"

In [0]:
# Function to list all the files in Silver layer in ADLS Gen2
def list_silver_files():
    display(dbutils.fs.ls("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/silver"))
    return "Silver files listed successfully"

In [0]:
# Function to list all the files in Gold layer in ADLS Gen2
def list_gold_files():
    display(dbutils.fs.ls("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/gold"))
    return "Gold files listed successfully"

In [0]:
# Function to read a file in bronze layer
def read_bronze_file_csv(file_name):
    data = spark.read.csv("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/bronze/" + file_name + ".csv", header=True, inferSchema=True)
    return data


In [0]:
# Function to read a file in Silver layer
def read_silver_file_csv(file_name):
    data = spark.read.csv("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/silver/" + file_name + ".csv", header=True, inferSchema=True)
    return data

In [0]:
# Function to read a file in Gold layer
def read_gold_file_csv(file_name):
    data = spark.read.csv("abfss://optum@adlsdataoptum.dfs.core.windows.net/medallion/gold/" + file_name + ".csv", header=True, inferSchema=True)
    return data

In [0]:
# Function for displaying the data
def display_data(df):
   return display(df.limit(10))


In [0]:
# Function to write or load the result into a Azure SQL database
def write_to_database(df, table_name):
    hostname = dbutils.secrets.get(scope = "optumScope", key = "AzureSqlHostName")
    port = dbutils.secrets.get(scope = "optumScope", key = "AzureSqlServerPort")
    database = dbutils.secrets.get(scope = "optumScope", key = "AzureSqlDatabaseName")
    DBProperties = {
            "user" : dbutils.secrets.get(scope = "optumScope", key = "AzureSqlUsername"),
            "password" : dbutils.secrets.get(scope = "optumScope", key = "AzureSqlPassword")
            }
    urlOfTarget = "jdbc:sqlserver://{0}:{1};database={2}".format(hostname, port, database)
    output = df.write.jdbc(url=urlOfTarget, table=table_name, mode="overwrite", properties=DBProperties)
    return "***** Successfully Written to Database******"